## Brochure Creator

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits

We will be provided with a company name and their primary website url

In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper_for_006 import fetch_website_contents, fetch_website_links
from openai import OpenAI

In [2]:
# Initializing the variables

load_dotenv(override=True)              # reads the .env file and loads the environment variables into this virtual environment
gemini_api_key = os.getenv('GEMINI_API_KEY')   # gets the GEMINI_API_KEY from the environment variables
gemini_base_url = os.getenv('GEMINI_BASE_URL') # gets the GEMINI_BASE_URL from the environment variables

# Validating the API Key
if gemini_api_key and gemini_api_key.startswith('AQ.') and len(gemini_api_key)>10:
    print("API Key looks good so far")
else:
    print("There might be a problem with the API Key")

# Assigning the MODEL
MODEL = 'gemini-3.1-flash-lite'

# Object for OpenAI class with gemini_api_key and gemini_base_url
openai = OpenAI(api_key=gemini_api_key, base_url = gemini_base_url)

API Key looks good so far


In [3]:
# Testing the model response through OpenAI endpoint

response = openai.chat.completions.create(
    model = MODEL,
    messages=[
        {'role':"system", 'content':'You are friendly assistant!'},
        {'role':'user', 'content':'Hello, I am Ram'}
    ]
)

print(response.choices[0].message.content)

Hello, Ram! It's a pleasure to meet you. How are you doing today? Is there anything I can help you with?


In [4]:
# Sample website url with various links inside the website (including relevant and irrelevant)

links = fetch_website_links('https://huggingface.co')
links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/storage',
 '/docs',
 '/enterprise',
 '/pricing',
 '/tasks',
 '/chat',
 '/collections',
 '/languages',
 '/organizations',
 '/blog',
 '/posts',
 '/papers',
 '/hardware',
 '/learn',
 '/join/discord',
 'https://discuss.huggingface.co/',
 'https://github.com/huggingface',
 '/enterprise',
 '/pro',
 '/support',
 '/inference/models',
 '/inference-endpoints',
 '/storage',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/thinkingmachines/Inkling',
 '/prism-ml/Ternary-Bonsai-27B-gguf',
 '/prism-ml/Bonsai-27B-gguf',
 '/baidu/Unlimited-OCR',
 '/zai-org/GLM-5.2',
 '/models',
 '/spaces/webml-community/bonsai-webgpu-kernels',
 '/spaces/Sneak-Moose/Pro-Realism-Edit-Studio',
 '/spaces/ICML-2026-agent-repro/challenge',
 '/spaces/kulkas2pintu/wan555',
 '/spaces/jasfn/LTX-2.3-10Eros',
 '/spaces',
 '/datasets/openbmb/UltraX-Preview',
 '/datasets/FlyRank/internship-warehouse',
 '/datasets/markov-ai/gaming-500-hours',
 '/datasets/SupraLabs/reasoning-corpus-4K-5M-v

### Step 1: Extract all the relevant links from the URL

Use a call to gemini-3.1-flash-lite to read all the links on a webpage and respond with relevant links in structured JSON format

It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

In [5]:
# System Prompt to obtain the useful or relevant links from all the links in the website

# Since we use 1 example to indicate the output format, it is a one-shot prompting technique

system_prompt_for_relevant_links = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""


In [6]:
# User prompt to obtain all the useful or relevant links by providing all the links 
# that are fetched from the fetch_website_links() by passing the website url (relevant and irrelevant links as input)

def get_user_prompt_for_relevant_links(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt



In [7]:
# validating System prompt and User prompt

print(f"System Prompt = {system_prompt_for_relevant_links}")
print("-------------------------------------------------------------------------------")
print(f"User Prompt = {get_user_prompt_for_relevant_links("https://huggingface.co")}")

System Prompt = 
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}

-------------------------------------------------------------------------------
User Prompt = 
Here is the list of links on the website https://huggingface.co -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/
/models
/datasets
/spaces
/storage
/docs
/enterprise
/pricing
/tasks
/chat
/collections
/languages
/organizations
/blog
/pos

In [8]:
# Function to call the model to select the relevant links from all the links using system prompt and user prompt

def select_the_relevant_links(url):
    response = openai.chat.completions.create(
        model = MODEL,
        messages = [
            {'role':'system', 'content':system_prompt_for_relevant_links},
            {'role':'user', 'content':get_user_prompt_for_relevant_links(url)}
        ],
        response_format = {'type':'json_object'}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [9]:
# Calling the model to select the relevant links for a webiste

select_the_relevant_links("https://huggingface.co")

{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'changelog', 'url': 'https://huggingface.co/changelog'},
  {'type': 'github', 'url': 'https://github.com/huggingface'},
  {'type': 'linkedin',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

#### Rewriting the model calling function for better user experience

In [10]:
# Function to call the model to select the relevant links from all the links using system prompt and user prompt

def select_the_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model = MODEL,
        messages = [
            {'role':'system', 'content':system_prompt_for_relevant_links},
            {'role':'user', 'content':get_user_prompt_for_relevant_links(url)}
        ],
        response_format = {'type':'json_object'}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f'Found {len(links['links'])} relevant links')
    return links

In [11]:
# calling the model for output with better user experience

select_the_relevant_links('https://huggingface.co')

# As the model output is dynamic, we might get different output for different runs i.e., different number of relevant links found for each run

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 7 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'github page', 'url': 'https://github.com/huggingface'},
  {'type': 'linkedin page',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

In [12]:
# calling the model for output with better user experience

select_the_relevant_links('https://huggingface.co')

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 6 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'linkedin page',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

### Step 2: Make the Brochure with the content from all the relevant links

Here, we assemble all the information we have obtained in Step 1 i.e., all the relevant links

Extract the contents from the landing page of the website

Extract the content of all the relevant links

Assemble the entire information and pass it to the gemini-3.1-flash-lite model through the user prompt to create the brochure

In [13]:
# To fetch the contents from landing page and from all the relevant links

def fetch_contents_from_landing_page_and_all_the_relevant_links(url):
    contents = fetch_website_contents(url)                              # Contents from the landing page i.e., https://huggingface.co
    relevant_links = select_the_relevant_links(url)                     # Obtaining all the relevant links
    result = f"## Landing Page: \n\n{contents}\n## Relevant Links: \n"  
    for link in relevant_links['links']:                                # Iterating over all the relevant links obtained from select_the_relevant_links(url) in json format
        result += f'\n\n### Link : {link['type']}\n'                    # Appending the relevant link type
        result += fetch_website_contents(link['url'])                   # Appending the content from relevant link url
    return result

# Result could be like this:
'''
## Landing Page:

Hugging Face - The AI community buidling the future.

## Relevant Links: 

### Link : 'about page'
We are on a mission to democratize good machine learning, one commit at a time.

### Link : 'enterprise page'
Give your organization the most advanced platform to build AI with enterprise-grade security, access controls, dedicated support and more.

### Link : 'careers page'
If you're interested in joining us, but don't tick every box, we still encourage you to apply! We're building a diverse team whose skills, experiences, and background complement one another. 

### Link : 'brand page'
......
......
......
......
......
'''

"\n## Landing Page:\n\nHugging Face - The AI community buidling the future.\n\n## Relevant Links: \n\n### Link : 'about page'\nWe are on a mission to democratize good machine learning, one commit at a time.\n\n### Link : 'enterprise page'\nGive your organization the most advanced platform to build AI with enterprise-grade security, access controls, dedicated support and more.\n\n### Link : 'careers page'\nIf you're interested in joining us, but don't tick every box, we still encourage you to apply! We're building a diverse team whose skills, experiences, and background complement one another. \n\n### Link : 'brand page'\n......\n......\n......\n......\n......\n"

In [14]:
print(fetch_contents_from_landing_page_and_all_the_relevant_links('https://huggingface.co'))

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 6 relevant links
## Landing Page: 

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
thinkingmachines/Inkling
Updated
about 17 hours ago
•
16.4k
•
1.3k
prism-ml/Ternary-Bonsai-27B-gguf
Updated
3 days ago
•
432k
•
876
prism-ml/Bonsai-27B-gguf
Updated
4 days ago
•
1.4M
•
555
baidu/Unlimited-OCR
Updated
18 days ago
•
2.24M
•
2.51k
zai-org/GLM-5.2
Updated
19 da

In [15]:
# System prompt to generate the brochure

system_prompt_to_generate_brochure = '''
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
'''

In [16]:
# Function to create the user prompt with all the information obtained from the landing page and relevant links contents

def get_user_prompt_to_generate_brochure(company_name, url):
    user_prompt = f'''
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
'''
    user_prompt += fetch_contents_from_landing_page_and_all_the_relevant_links(url)
    user_prompt = user_prompt[:5000]                # truncate the prompt if more than 5000 characters
    return user_prompt

In [17]:
# Validating the system prompt and user prompt to generate the brochure

print(f'System Prompt: {system_prompt_to_generate_brochure}')
print('---------------------------------------------------------------------------------')
print(f'User Prompt: {get_user_prompt_to_generate_brochure('HuggingFace', 'https://huggingface.co')}')

System Prompt: 
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.

---------------------------------------------------------------------------------
Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 8 relevant links
User Prompt: 
You are looking at a company called: HuggingFace
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.


## Landing Page: 

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Pap

In [18]:
# Function to call the model to create a brochure for the company

def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model = MODEL,
        messages=[
            {'role':'system', 'content':system_prompt_to_generate_brochure},
            {'role':'user', 'content':get_user_prompt_to_generate_brochure(company_name,url)}
        ]
    )
    result = response.choices[0].message.content
    display(Markdown(result))

    ''' 
    result              - is the text returned by the model
    Markdown(result)    - tells the Jupyter to treat the text as Markdown
    display()           - renders it nicely in the notebook output area
    '''

### Final Output

In [19]:
# Calling the function to generate the brochure

create_brochure('HuggingFace', 'https://huggingface.co')

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 8 relevant links


# Hugging Face: The AI Community Building the Future

## About the Company
Hugging Face is the leading global platform for the machine learning community. Often described as the "Home of Machine Learning," the company serves as the central hub where researchers, developers, and organizations collaborate to build, share, and deploy artificial intelligence. By democratizing access to state-of-the-art tools, Hugging Face is fundamentally shaping the future of open-source AI.

## Our Platform
We provide a massive, collaborative ecosystem that hosts millions of assets, enabling users to move faster in their AI development cycles:
*   **Models:** Browse over 2 million machine learning models spanning every imaginable task.
*   **Datasets:** Access over 500,000 datasets to fuel your training and research.
*   **Spaces:** Deploy and explore over 1 million AI applications directly in your browser.
*   **Infrastructure:** Benefit from robust tools including Inference Endpoints, Storage Buckets, and hardware support to scale from individual projects to enterprise-grade solutions.

## Community & Culture
Hugging Face is built on the philosophy of open collaboration. Our culture is deeply rooted in the open-source movement, fostering a global environment where innovation is shared rather than siloed. 
*   **Collaboration:** From daily papers to community-led forums and Discord channels, we provide the space for the brightest minds in AI to exchange ideas.
*   **Learning:** We are committed to education, offering resources like "Hugging Face Fundamentals" to help the next generation of engineers master machine learning.
*   **Engagement:** Our community is active and vibrant, with constant contributions to model updates, documentation, and research, ensuring that the platform remains at the bleeding edge of technology.

## For Enterprises
We empower businesses to leverage the power of open-source AI with professional-grade solutions:
*   **Enterprise Support:** Tailored assistance for teams integrating AI into their infrastructure.
*   **Inference Providers:** Scalable deployment options that turn models into production-ready applications.
*   **Security & Scalability:** Advanced tools for managing private models and datasets within a secure, collaborative framework.

## Careers
Hugging Face is consistently seeking passionate, mission-driven individuals to join our team. We are a distributed, international organization that values intellectual curiosity, collaborative spirit, and a deep interest in the rapidly evolving landscape of AI and machine learning. If you are interested in building the infrastructure that supports the global AI research community, we invite you to connect with us on our GitHub or reach out through our official channels.

---

**Join the movement.**
Explore the platform, discover new models, and start building at [huggingface.co](https://huggingface.co).

## Small Tweak to the Final Output to make the user experience better

Here, we are streaming the data in the output instead of pushing all at once to improve the user experience

In [20]:
# To stream the output

def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model = MODEL,
        messages = [
            {'role':'system', 'content':'system_prompt_to_generate_brochure'},
            {'role':'user', 'content':get_user_prompt_to_generate_brochure(company_name, url)},
        ],
        stream = True                                       # Tells the API to return the response chunk by chunk
    )
    response = ""
    # Markdown("") -creates an empty markdown output area
    # display_id=True -let's you update that same area later instead of printing new blocks
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''    # If no content, then add empty string '' to the response
        update_display(Markdown(response), display_id=display_handle.display_id)        # updates the notebook with the accumulated response so far

''' 
In short:
display_handle = display(Markdown(""), display_id=True)                     -- Creates an empty markdown display area in Jupyter notebook

update_display(Markdown(response), display_id=display_handle.display_id)    -- For every time a new chunk is added to the response, it updates the markdown display that was already created earlier

This makes it look like the answer is being generated live
'''
        

' \nIn short:\ndisplay_handle = display(Markdown(""), display_id=True)                     -- Creates an empty markdown display area in Jupyter notebook\n\nupdate_display(Markdown(response), display_id=display_handle.display_id)    -- For every time a new chunk is added to the response, it updates the markdown display that was already created earlier\n\nThis makes it look like the answer is being generated live\n'

In [21]:
# Calling the function to invoke the model to stream the output data

stream_brochure('HuggingFace', 'https://huggingface.co')

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 6 relevant links


# Hugging Face: The AI Community Building the Future

## Who We Are
Hugging Face is the leading global platform for the machine learning community. We serve as the central hub where developers, researchers, and organizations come together to create, discover, and collaborate on the next generation of artificial intelligence. Our mission is to democratize AI by providing an open, accessible ecosystem for shared innovation.

## Our Platform at a Glance
Hugging Face offers a comprehensive suite of tools designed to accelerate the machine learning development lifecycle:

* Models: Explore over 2 million pre-trained models for diverse tasks.
* Datasets: Access over 500,000 datasets to power your training and research.
* Spaces: Showcase and test machine learning applications in real-time.
* HuggingChat: Engage with advanced conversational AI directly on our platform.
* Storage Buckets: Scale your projects with integrated, secure infrastructure.

## Why Choose Hugging Face?
We empower individuals and enterprises to move faster. Whether you are an independent researcher or a large organization, our platform provides the collaborative infrastructure you need to succeed:

* Collaborative Environment: Host and share public or private models, datasets, and apps with your team.
* Enterprise-Ready: We offer robust Enterprise Support, Inference Endpoints, and specialized hardware solutions to ensure your production workloads are secure and efficient.
* Expert Resources: Stay ahead of the curve with our community-driven blog, daily research papers, and extensive documentation.
* Global Network: Join a vibrant community of over 100,000 active members contributing to the future of open-source AI.

## Professional Solutions
For those looking to take their projects to the next level, we offer:
* Hugging Face PRO: Enhanced features for power users.
* Team & Enterprise Plans: Tailored solutions for large-scale development and organizational management.
* Inference Providers: Seamless integration with top-tier compute resources to deploy your models with ease.

## Join the Movement
The future of technology is being built in the open. Whether you are looking to browse the latest breakthroughs in machine learning, deploy your first AI agent, or build an enterprise-scale solution, Hugging Face provides the tools to make it happen.

Visit us at huggingface.co to get started today.

### Changing the System Prompt to create the brochure in a Funny way

In [22]:
system_prompt_to_generate_brochure = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [23]:
def get_user_prompt_to_generate_brochure(company_name, url):
    user_prompt = f'''
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
'''
    user_prompt += fetch_contents_from_landing_page_and_all_the_relevant_links(url)
    user_prompt = user_prompt[:5000]                # truncate the prompt if more than 5000 characters
    return user_prompt

In [24]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model = MODEL,
        messages = [
            {'role':'system', 'content':'system_prompt_to_generate_brochure'},
            {'role':'user', 'content':get_user_prompt_to_generate_brochure(company_name, url)},
        ],
        stream = True                                       # Tells the API to return the response chunk by chunk
    )
    response = ""
    # Markdown("") -creates an empty markdown output area
    # display_id=True -let's you update that same area later instead of printing new blocks
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''    # If no content, then add empty string '' to the response
        update_display(Markdown(response), display_id=display_handle.display_id)        # updates the notebook with the accumulated response so far


In [25]:
stream_brochure('HuggingFace', 'https://huggingface.co')

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 8 relevant links


# Hugging Face: The AI Community Building the Future

Hugging Face is the global collaboration platform for the machine learning community. Designed to foster innovation, it serves as the central hub where developers, researchers, and organizations come together to create, discover, and deploy artificial intelligence models, datasets, and applications.

## What We Do
Hugging Face empowers the machine learning ecosystem by providing the infrastructure needed to move faster and build more effectively. Whether you are an individual researcher or a large enterprise, our platform offers the tools to collaborate on the next generation of AI.

### Core Pillars of the Platform
* **Models:** Explore a library of over 2 million machine learning models, from state-of-the-art LLMs to specialized computer vision and audio tools.
* **Datasets:** Access over 500,000 datasets to train, fine-tune, and test your models.
* **Spaces:** Showcase your work with over 1 million interactive AI applications. Run, test, and share your demos directly in the browser.
* **Infrastructure:** Benefit from enterprise-grade support, inference endpoints, and storage buckets designed to scale with your project needs.

## Why Join Hugging Face?
We believe the future of AI is built through openness and collaboration. By hosting your projects on Hugging Face, you become part of a massive, vibrant community of over 100,000 active contributors.

* **Collaborate:** Work together with teams and organizations on public or private projects.
* **Innovate:** Stay at the forefront of the industry with access to the latest research, daily papers, and technical blog posts.
* **Learn:** Grow your skills through community forums, Discord channels, and dedicated educational tracks.
* **Scale:** Utilize our robust Enterprise solutions to bring your research from prototype to production with confidence.

## Trusted by the Industry
From open-source enthusiasts to global enterprises, Hugging Face provides the tools to manage the entire AI lifecycle. With integrated solutions for inference providers, hardware optimization, and model hosting, we are the home of machine learning.

## Get Started Today
Join the movement at [huggingface.co](https://huggingface.co). 

Whether you are looking to browse the latest trending models, contribute to a dataset, or deploy your first AI application in a Space, Hugging Face gives you the platform to turn your ideas into reality. 

**Hugging Face: The AI community building the future.**